# 🌤️ Weather Classification — End-to-End ML Pipeline
**Senior Data Science Project** | Dataset: 13,200 records | 4 Weather Classes

> Target: Prediksi tipe cuaca (Sunny, Cloudy, Rainy, Snowy) dari fitur meteorologi

## 📦 0. Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import joblib, warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded ✅")

## 📊 1. Load Data

In [ ]:
# Load dataset
df = pd.read_csv("../data/cleaned_dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 🔍 2. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset info & missing values
print("=== Data Types ===")
print(df.dtypes)
print(f"
=== Missing Values ===")
print(df.isnull().sum())
print(f"
=== Duplicates: {df.duplicated().sum()}")
df.describe()

In [ ]:
# Target distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
vc = df["Weather Type"].value_counts()
ax1.bar(vc.index, vc.values, color=["#F4A261","#457B9D","#2A9D8F","#E9C46A"])
ax1.set_title("Weather Type Distribution", fontsize=14, fontweight="bold")
ax1.set_ylabel("Count")
ax2.pie(vc.values, labels=vc.index, autopct="%1.1f%%",
        colors=["#F4A261","#457B9D","#2A9D8F","#E9C46A"], startangle=90)
ax2.set_title("Weather Type Proportion", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()
print("Dataset perfectly balanced! Each class has", vc[0], "records.")

## ⚙️ 3. Data Preprocessing

In [ ]:
# --- 3.1 Handle Outliers (IQR Method) ---
num_cols = ["Temperature","Humidity","Wind Speed","Precipitation (%)",
            "Atmospheric Pressure","UV Index","Visibility (km)"]
df_clean = df.copy()

outliers_removed = {}
for col in num_cols:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    before = df_clean[col].between(lo, hi).sum()
    df_clean[col] = df_clean[col].clip(lo, hi)
    outliers_removed[col] = len(df_clean) - before

print("Outliers clipped per column:")
for k, v in outliers_removed.items():
    print(f"  {k}: {v} values clipped")

In [ ]:
# --- 3.2 Label Encoding ---
le_cloud    = LabelEncoder().fit(df_clean["Cloud Cover"])
le_season   = LabelEncoder().fit(df_clean["Season"])
le_location = LabelEncoder().fit(df_clean["Location"])
le_target   = LabelEncoder().fit(df_clean["Weather Type"])

df_clean["Cloud_Cover_enc"]   = le_cloud.transform(df_clean["Cloud Cover"])
df_clean["Season_enc"]        = le_season.transform(df_clean["Season"])
df_clean["Location_enc"]      = le_location.transform(df_clean["Location"])
df_clean["Weather_Type_enc"]  = le_target.transform(df_clean["Weather Type"])

print("Encoding mapping:")
print(f"  Cloud Cover: {dict(zip(le_cloud.classes_, le_cloud.transform(le_cloud.classes_)))}")
print(f"  Season:      {dict(zip(le_season.classes_, le_season.transform(le_season.classes_)))}")
print(f"  Location:    {dict(zip(le_location.classes_, le_location.transform(le_location.classes_)))}")
print(f"  Target:      {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

In [ ]:
# --- 3.3 Feature Engineering ---
# Interaction feature: Humidity x Precipitation
df_clean["Humidity_x_Precip"] = df_clean["Humidity"] * df_clean["Precipitation (%)"] / 100
# Ratio: Temperature per UV intensity
df_clean["Temp_UV_ratio"]     = df_clean["Temperature"] / (df_clean["UV Index"] + 1)

print("New engineered features:")
print(df_clean[["Humidity_x_Precip","Temp_UV_ratio"]].describe())

In [ ]:
# --- 3.4 Feature Selection & Scaling ---
features = ["Temperature","Humidity","Wind Speed","Precipitation (%)",
            "Atmospheric Pressure","UV Index","Visibility (km)",
            "Cloud_Cover_enc","Season_enc","Location_enc",
            "Humidity_x_Precip","Temp_UV_ratio"]

X = df_clean[features]
y = df_clean["Weather_Type_enc"]

scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

# --- 3.5 Train-Test Split (80:20) ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

## 🤖 4. Modeling
### 4.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("Logistic Regression trained ✅")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")

### 4.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest trained ✅")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

## 📈 5. Evaluasi Model

In [ ]:
def evaluate(name, yt, yp, classes):
    print(f"
{"="*50}")
    print(f"  {name}")
    print(f"{"="*50}")
    print(f"  Accuracy : {accuracy_score(yt,yp):.4f}")
    print(f"  Precision: {precision_score(yt,yp,average="weighted"):.4f}")
    print(f"  Recall   : {recall_score(yt,yp,average="weighted"):.4f}")
    print(f"  F1-Score : {f1_score(yt,yp,average="weighted"):.4f}")
    print("
Classification Report:")
    print(classification_report(yt, yp, target_names=classes))

classes = le_target.classes_
evaluate("Logistic Regression", y_test, y_pred_lr, classes)
evaluate("Random Forest",       y_test, y_pred_rf, classes)

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, yp, title in zip(axes, [y_pred_lr, y_pred_rf],
                          ["Logistic Regression", "Random Forest"]):
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=classes, yticklabels=classes, linewidths=0.5)
    ax.set_title(f"Confusion Matrix — {title}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("../visualizations/05_confusion_matrices.png", bbox_inches="tight")
plt.show()

## 🧠 6. Feature Importance & Interpretasi

In [ ]:
# Random Forest Feature Importance
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = ["#e63946" if v > feat_imp.median() else "#457B9D" for v in feat_imp.sort_values(ascending=True)]
feat_imp.sort_values(ascending=True).plot(kind="barh", ax=ax, color=colors_bar)
ax.axvline(feat_imp.median(), color="gray", linestyle="--", alpha=0.7, label="Median")
ax.set_title("Feature Importance — Random Forest", fontsize=14, fontweight="bold")
ax.set_xlabel("Importance Score")
ax.legend()
plt.tight_layout()
plt.savefig("../visualizations/06_feature_importance.png", bbox_inches="tight")
plt.show()

print("
📊 Top-5 Most Important Features:")
for i, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
    print(f"  {i}. {feat}: {imp:.4f} ({imp*100:.1f}%)")

### 💡 Insight: Kenapa Fitur Ini Penting?

| Fitur | Insight |
|---|---|
| **Precipitation (%)** | Prediktor terkuat Rainy. Curah hujan tinggi → Rainy hampir pasti |
| **Temperature** | Membedakan Snowy (< 0°C) vs Sunny (> 25°C) secara tajam |
| **Humidity** | Korelasi kuat dengan Rainy & Cloudy. Humidity > 80% → bukan Sunny |
| **Humidity × Precipitation** | Interaksi engineered — memperkuat sinyal wet weather |
| **Cloud Cover** | Discriminator langsung untuk Cloudy vs Clear sky |
| **UV Index** | Tinggi → Sunny. Rendah/0 → Snowy atau Rainy |
| **Atmospheric Pressure** | Tekanan rendah (<1000 hPa) → cuaca ekstrem (Rainy/Snowy) |

## 💾 7. Save Models

In [ ]:
import joblib
joblib.dump(rf,          "../model/random_forest_model.pkl")
joblib.dump(lr,          "../model/logistic_regression_model.pkl")
joblib.dump(scaler,      "../model/scaler.pkl")
joblib.dump(le_target,   "../model/label_encoder_target.pkl")
joblib.dump(le_cloud,    "../model/label_encoder_cloud.pkl")
joblib.dump(le_season,   "../model/label_encoder_season.pkl")
joblib.dump(le_location, "../model/label_encoder_location.pkl")
joblib.dump(features,    "../model/feature_names.pkl")
print("All models saved! ✅")